# Bài 26: Sự hình thành ảnh (Image Formation)

Cho đến giờ chúng ta đã xử lý ảnh mà không nghĩ xem chúng từ đâu mà có. Bài này giải mã bí ẩn đó bằng cách xem xét: **hình học** (ma trận chiếu được suy ra thế nào từ mô hình camera lỗ kim), **quang học** (một thấu kính thực làm gì mà mô hình camera lỗ kim không mô tả được &mdash; lấy nét và nhòe), và **cảm biến/màu sắc** (một cảm biến thực chỉ ghi nhận một màu cho mỗi điểm ảnh và cần tái tạo phần còn lại như thế nào, cùng với hai cạm bẫy kinh điển &mdash; gamma và cân bằng trắng &mdash; hay làm vấp ngã các thao tác xử lý ảnh ngây thơ).

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Suy ra ma trận chiếu lỗ kim

Một camera lỗ kim (pinhole camera) chỉ cho đi qua các tia sáng đi qua một điểm cụ thể (gọi là *tâm chiếu (center of projection)*). Theo tam giác đồng dạng, một điểm 3D $(X, Y, Z)$ (tọa độ lấy camera làm tâm) rơi lên mặt phẳng ảnh tại

$$x = f\frac{X}{Z}, \qquad y = f\frac{Y}{Z}$$

trong đó $f$ là khoảng cách từ lỗ kim tới mặt phẳng ảnh. Chuyển sang đơn vị điểm ảnh (tính đến kích thước điểm ảnh và vị trí gốc tọa độ) đòi hỏi các **tham số nội (intrinsics)** $K$; và đưa vào vị trí và hướng của camera so với thế giới đòi hỏi các **tham số ngoại (extrinsics)** $[R | t]$. Gộp lại, trong tọa độ thuần nhất:

$$\underbrace{\begin{bmatrix}u\\v\\1\end{bmatrix}}_{\text{pixel}} \;\propto\; \underbrace{\begin{bmatrix}f_x&0&c_x\\0&f_y&c_y\\0&0&1\end{bmatrix}}_{K}\underbrace{\begin{bmatrix}R & t\end{bmatrix}}_{\text{extrinsics}}\begin{bmatrix}X\\Y\\Z\\1\end{bmatrix}_{\text{world}}$$

Đây chính xác là $P = K [R | t]$ mà các Bài 27-29 (Hiệu chỉnh camera, Hình học epipolar và Cấu trúc từ chuyển động) sẽ dùng. Mọi ký hiệu đều có thể truy về một nguyên nhân vật lý: $f_x, f_y$ ứng với tiêu cự và kích thước điểm ảnh; $(c_x, c_y)$ ứng với nơi trục quang học chạm vào cảm biến (gọi là *điểm chính (principal point)*); $R, t$ ứng với tư thế (pose) của camera.

In [ ]:
K = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1]], dtype=np.float64)
R, t = np.eye(3), np.array([0, 0, 5.0])
P = K @ np.hstack([R, t.reshape(3, 1)])

cube_corners = np.array([[x, y, z] for x in (-1, 1) for y in (-1, 1) for z in (-1, 1)], dtype=np.float64)
homogeneous = np.hstack([cube_corners, np.ones((8, 1))])
projected = (P @ homogeneous.T).T
pixels = projected[:, :2] / projected[:, 2:3]

edges = [(0, 1), (0, 2), (0, 4), (1, 3), (1, 5), (2, 3), (2, 6),
         (3, 7), (4, 5), (4, 6), (5, 7), (6, 7)]

plt.figure(figsize=(5, 4))
for i, j in edges:
    plt.plot([pixels[i, 0], pixels[j, 0]], [pixels[i, 1], pixels[j, 1]], color='tab:blue')
plt.scatter(*pixels.T, color='red', zorder=5)
plt.gca().invert_yaxis()
plt.title('A 3D cube, projected through P = K[R|t]')
plt.show()

## Thấu kính thực: lấy nét và nhòe do mất nét

Camera lỗ kim cho vào quá ít ánh sáng đến mức cần thời gian phơi sáng dài một cách vô lý. Vì vậy, camera thực dùng thấu kính để thu nhiều ánh sáng hơn hẳn. Tuy nhiên, thấu kính chỉ lấy nét sắc cho các vật ở một khoảng cách cụ thể (gọi là **mặt phẳng tiêu (focal plane)**). Các điểm gần hơn hoặc xa hơn khoảng đó trải ánh sáng của chúng lên một đĩa nhỏ trên cảm biến (gọi là **vòng tròn nhòe (circle of confusion)**) thay vì tại một điểm &mdash; chính là các kernel tích chập làm mờ ở Bài 9-10, chỉ là áp dụng với kích thước kernel phụ thuộc vào độ sâu.

In [ ]:
scene = np.zeros((200, 300, 3), dtype=np.uint8)
cv2.rectangle(scene, (20, 20), (90, 180), (255, 120, 30), -1)    # near object
cv2.rectangle(scene, (110, 20), (190, 180), (30, 200, 255), -1)  # object at the focal plane
cv2.rectangle(scene, (210, 20), (280, 180), (120, 255, 60), -1)  # far object

# A letter on each block, drawn into the scene itself (so it blurs along with
# the block), gives the eye something with fine detail to judge sharpness by --
# a flat color block would hardly show any blur at all in its interior.
for cx, cy in [(55, 100), (150, 100), (245, 100)]:
    (tw, th), _ = cv2.getTextSize('E', cv2.FONT_HERSHEY_SIMPLEX, 2.6, 6)
    org = (cx - tw // 2, cy + th // 2)
    cv2.putText(scene, 'E', org, cv2.FONT_HERSHEY_SIMPLEX, 2.6, (0, 0, 0), 6, cv2.LINE_AA)

depths_m = {'near': 2.0, 'mid': 5.0, 'far': 9.0}
focus_depth_m = 5.0

def defocus_kernel_size(depth, focus, strength=15.0):
    radius = strength * abs(depth - focus) / focus
    ksize = int(radius) * 2 + 1
    return max(1, ksize)

def blur_with_context(image, y0, y1, x0, x1, ksize, pad):
    """Blur a region using a padded crop, so the blur can pull in real neighboring
    pixels (including the background) instead of only the region's own solid color."""
    if ksize <= 1:
        return image[y0:y1, x0:x1].copy()
    yy0, yy1 = max(0, y0 - pad), min(image.shape[0], y1 + pad)
    xx0, xx1 = max(0, x0 - pad), min(image.shape[1], x1 + pad)
    blurred = cv2.GaussianBlur(image[yy0:yy1, xx0:xx1], (ksize, ksize), 0)
    return blurred[y0 - yy0:y0 - yy0 + (y1 - y0), x0 - xx0:x0 - xx0 + (x1 - x0)]

defocused = scene.copy()
regions = {'near': (20, 180, 20, 90), 'mid': (20, 180, 110, 190), 'far': (20, 180, 210, 280)}
for name, (y0, y1, x0, x1) in regions.items():
    k = defocus_kernel_size(depths_m[name], focus_depth_m)
    print(f'{name:>5} object: depth={depths_m[name]}m, blur kernel size={k}')
    defocused[y0:y1, x0:x1] = blur_with_context(scene, y0, y1, x0, x1, k, pad=k)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
axes[0].set_title('All in focus (pinhole idealization)')
axes[1].imshow(cv2.cvtColor(defocused, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Focused at {focus_depth_m}m\n(near/far objects blurred)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Điều này giải thích tại sao khẩu độ rộng hơn thu được nhiều ánh sáng hơn (với cái giá là độ sâu trường ảnh (depth of field) nông hơn), trong khi khẩu độ hẹp cho ra ảnh sắc nét (nhưng cho vào ít ánh sáng hơn) &mdash; sự đánh đổi khẩu độ/độ sâu trường ảnh kinh điển trong nhiếp ảnh. Đây là lý do ảnh chân dung thường làm mờ phông nền trong khi giữ chủ thể sắc nét.

Ảnh động này quét khoảng cách lấy nét từ 1m đến gần 10m rồi quay lại, tính lại kernel mất nét của mỗi khối ngay lập tức, với một chữ cái vẽ trên mỗi khối để dễ đánh giá độ sắc nét bằng mắt. Một thanh trượt bên dưới ảnh đánh dấu độ sâu thực của mỗi vật bằng một chấm màu tương ứng với khối của nó; hình tam giác di chuyển theo dõi khoảng cách lấy nét hiện tại, và chấm nào gần nó nhất là vật đang được lấy nét sắc -- vòng tròn nhòe được trực quan hóa trực tiếp khi nó lớn lên và thu nhỏ lại.

<img src="../animations/defocus_blur_demo.gif" alt="Ảnh động nhòe do mất nét: mặt phẳng tiêu quét từ gần đến xa rồi quay lại, với mỗi vật trong ba vật có chữ cái trở nên sắc nét khi điểm đánh dấu lấy nét của thanh trượt đi qua độ sâu của nó và bị nhòe trong các trường hợp khác" style="display: block; margin: 0 auto; max-width: 680px;">

## Mảng lọc màu Bayer: một màu cho mỗi điểm ảnh

Để giảm chi phí, hầu hết cảm biến camera không đo riêng đỏ, lục và lam tại mọi điểm ảnh. Thay vào đó, một mảng cảm quang đơn sắc duy nhất nằm dưới một bức khảm các bộ lọc màu nhỏ xíu, nên mỗi điểm ảnh riêng lẻ về mặt vật lý chỉ ghi nhận *một* trong ba màu. Cách sắp xếp phổ biến nhất (RGGB) &mdash; **bộ lọc Bayer (Bayer filter)** (<a href="../references.html#bayer-1976">Bayer, 1976</a>) &mdash; lặp lại theo mẫu ô 2x2: một bộ lọc đỏ, một bộ lọc lam, và *hai* bộ lọc lục, vì mắt người nhạy nhất với màu lục. Do đó, theo cách cấu tạo, mọi điểm ảnh trong đầu ra cảm biến thô đều thiếu hai phần ba thông tin màu của nó.

In [ ]:
rng = np.random.default_rng(0)
bayer_scene = np.zeros((120, 160, 3), dtype=np.uint8)
cv2.rectangle(bayer_scene, (10, 10), (70, 100), (40, 180, 220), -1)
cv2.circle(bayer_scene, (110, 60), 40, (200, 90, 40), -1)
bayer_scene = np.clip(bayer_scene.astype(np.float64) + rng.normal(0, 4, bayer_scene.shape), 0, 255).astype(np.uint8)

def make_bayer_rggb(img):
    # (row, col) parity: (even,even)=R, (even,odd)=G, (odd,even)=G, (odd,odd)=B
    H, W = img.shape[:2]
    mosaic = np.zeros((H, W), dtype=np.uint8)
    mosaic[0::2, 0::2] = img[0::2, 0::2, 2]  # R  (img is BGR, so channel 2 = R)
    mosaic[0::2, 1::2] = img[0::2, 1::2, 1]  # G
    mosaic[1::2, 0::2] = img[1::2, 0::2, 1]  # G
    mosaic[1::2, 1::2] = img[1::2, 1::2, 0]  # B
    return mosaic

def colorize_bayer_rggb(mosaic):
    """Put each pixel's raw reading into its OWN filter's color channel, zeroing the
    other two -- this is what actually lets you see the RGGB pattern, since a plain
    grayscale view of the mosaic just shows intensities with no color information at all."""
    H, W = mosaic.shape
    colored = np.zeros((H, W, 3), dtype=np.uint8)
    colored[0::2, 0::2, 0] = mosaic[0::2, 0::2]  # R filter
    colored[0::2, 1::2, 1] = mosaic[0::2, 1::2]  # G filter
    colored[1::2, 0::2, 1] = mosaic[1::2, 0::2]  # G filter
    colored[1::2, 1::2, 2] = mosaic[1::2, 1::2]  # B filter
    return colored

bayer = make_bayer_rggb(bayer_scene)
bayer_colored = colorize_bayer_rggb(bayer)

y0, y1, x0, x1 = 40, 56, 20, 36  # a 16x16 crop, fully inside a colored shape

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(cv2.cvtColor(bayer_scene, cv2.COLOR_BGR2RGB))
axes[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, edgecolor='white', facecolor='none', linewidth=1.5))
axes[0].set_title('true scene (each pixel: R, G, and B)')
axes[1].imshow(bayer[y0:y1, x0:x1], cmap='gray')
axes[1].set_title('raw mosaic\n(grayscale readout -- no color info visible)')
axes[2].imshow(bayer_colored[y0:y1, x0:x1])
axes[2].set_title('same patch, colorized by filter\n(the actual RGGB pattern)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Khử khảm: tái tạo RGB từ bức khảm

**Khử khảm (demosaicking)** (hay "debayering") điền hai màu còn thiếu của mỗi điểm ảnh bằng cách nội suy từ các lân cận cùng màu — phiên bản đơn giản nhất là song tuyến tính: lấy trung bình các mẫu đã biết gần đó của một màu, đánh trọng số theo khoảng cách, chính là nội suy song tuyến tính ở Bài 9, chỉ là nội suy trên một lưới thưa, có mẫu hình, gồm các mẫu đã biết thay vì một ảnh dày đặc đã giảm mẫu.

In [ ]:
def demosaick_bilinear_rggb(mosaic):
    H, W = mosaic.shape
    R = np.zeros((H, W)); G = np.zeros((H, W)); B = np.zeros((H, W))
    R[0::2, 0::2] = mosaic[0::2, 0::2]
    G[0::2, 1::2] = mosaic[0::2, 1::2]; G[1::2, 0::2] = mosaic[1::2, 0::2]
    B[1::2, 1::2] = mosaic[1::2, 1::2]

    Rmask = np.zeros((H, W)); Rmask[0::2, 0::2] = 1
    Gmask = np.zeros((H, W)); Gmask[0::2, 1::2] = 1; Gmask[1::2, 0::2] = 1
    Bmask = np.zeros((H, W)); Bmask[1::2, 1::2] = 1

    def fill(channel, mask):
        # weighted average of known same-color neighbors within a 3x3 window
        kernel = np.array([[0.25, 0.5, 0.25], [0.5, 1.0, 0.5], [0.25, 0.5, 0.25]])
        weight_sum = cv2.filter2D(mask, -1, kernel, borderType=cv2.BORDER_REFLECT)
        value_sum = cv2.filter2D(channel * mask, -1, kernel, borderType=cv2.BORDER_REFLECT)
        filled = channel.copy()
        empty = mask == 0
        filled[empty] = value_sum[empty] / np.clip(weight_sum[empty], 1e-6, None)
        return filled

    R_full, G_full, B_full = fill(R, Rmask), fill(G, Gmask), fill(B, Bmask)
    return np.clip(np.stack([B_full, G_full, R_full], axis=-1), 0, 255).astype(np.uint8)

manual_demosaick = demosaick_bilinear_rggb(bayer)
cv2_demosaick = cv2.cvtColor(bayer, cv2.COLOR_BayerBG2BGR)

manual_err = np.abs(manual_demosaick.astype(np.float64) - bayer_scene.astype(np.float64)).mean()
cv2_err = np.abs(cv2_demosaick.astype(np.float64) - bayer_scene.astype(np.float64)).mean()
print(f'manual bilinear demosaick MAE vs ground truth: {manual_err:.2f}')
print(f'cv2 demosaick MAE vs ground truth:              {cv2_err:.2f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [bayer_scene, manual_demosaick, cv2_demosaick],
                          ['ground truth', 'manual bilinear\ndemosaick', 'cv2 demosaick']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Ảnh động này phóng to vào cùng vùng cắt khảm 16x16 và điền vào từng điểm ảnh, theo thứ tự quét mành, thay màu đã biết duy nhất của mỗi điểm ảnh bằng giá trị RGB cuối cùng sau khử khảm song tuyến tính -- hãy quan sát bức khảm thưa, một kênh được điền dần thành kết quả đầy đủ màu hiển thị bên phải.

<img src="../animations/bayer_demosaic_demo.gif" alt="Ảnh động khử khảm song tuyến tính, điền từng điểm ảnh của một vùng cắt khảm Bayer 16x16 cho đến khi nó khớp với ảnh màu đã khử khảm hoàn toàn" style="display: block; margin: 0 auto; max-width: 680px;">

Cả hai phương pháp đều tái tạo sát dữ liệu chuẩn (sai số còn lại chủ yếu chỉ là nhiễu cảm biến vốn đã có trong bức khảm, không phải nhiễu do khử khảm) — một phép điền song tuyến tính tự viết và phép khử khảm có sẵn của OpenCV chỉ chênh nhau một phần nhỏ của một giá trị điểm ảnh. Các thuật toán khử khảm thực tế tinh vi hơn song tuyến tính thuần túy (nội suy có nhận biết biên để tránh làm nhòe qua ranh giới vật thể, vì khử khảm song tuyến tính ngây thơ chính là thứ gây ra nhiễu viền màu "dây kéo (zipper)" thấy dọc theo các cạnh sắc ở camera rẻ tiền), nhưng ý tưởng cốt lõi — nội suy mỗi màu còn thiếu từ các lân cận cùng màu gần nhất — là như nhau.

### Cạm bẫy: dùng sai mẫu khảm

Cách sắp xếp R/G/B có ý nghĩa, và có nhiều hơn một quy ước (RGGB, BGGR, GRBG, GBRG, tùy cảm biến). Khử khảm với mẫu giả định *sai* không thất bại một cách êm thấm — nó âm thầm tạo ra một ảnh trông có vẻ hợp lý nhưng sai màu nghiêm trọng, vì mọi điểm ảnh đều nhận được giá trị, chỉ là giá trị sai.

In [ ]:
wrong_demosaick = cv2.cvtColor(bayer, cv2.COLOR_BayerRG2BGR)  # wrong pattern assumption
wrong_err = np.abs(wrong_demosaick.astype(np.float64) - bayer_scene.astype(np.float64)).mean()
print(f'correct pattern (BayerBG2BGR) MAE: {cv2_err:.2f}')
print(f'WRONG pattern (BayerRG2BGR) MAE:   {wrong_err:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(6, 3.5))
axes[0].imshow(cv2.cvtColor(cv2_demosaick, cv2.COLOR_BGR2RGB)); axes[0].set_title('correct pattern', fontsize=9)
axes[1].imshow(cv2.cvtColor(wrong_demosaick, cv2.COLOR_BGR2RGB)); axes[1].set_title('wrong pattern', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Sai số tăng vọt hơn một bậc độ lớn nếu dùng sai mẫu &mdash; mọi mẫu đỏ bị coi là lam và ngược lại, tạo ra một ảnh lệch màu mạnh. Đây là lý do thực tế mà các quy trình xử lý ảnh RAW luôn cần biết chính xác bố cục cảm biến của camera.

## Hiệu chỉnh gamma: giá trị điểm ảnh không phải ánh sáng tuyến tính

Camera và màn hình không lưu độ sáng tỉ lệ thuận với cường độ ánh sáng vật lý (**bức xạ tuyến tính (linear radiance)**). Thay vào đó, các giá trị được **mã hóa gamma**: `encoded = linear ** (1/gamma)` (thường gamma &asymp; 2.2, gần với chuẩn sRGB), phân bổ nhiều mức mã hóa hơn cho các tông tối &mdash; khớp với việc thị giác con người nhạy với vùng tối hơn vùng sáng, và về mặt lịch sử khớp với cách màn hình CRT phản hồi với điện áp. Giải mã thì đảo ngược nó: `linear = encoded ** gamma`.

In [ ]:
def encode_gamma(linear, gamma=1 / 2.2):
    return np.clip(linear, 0, 1) ** gamma

def decode_gamma(encoded, gamma=2.2):
    return np.clip(encoded, 0, 1) ** gamma

linear_values = np.linspace(0, 1, 100)
plt.plot(linear_values, linear_values, '--', color='gray', label='identity (no gamma)')
plt.plot(linear_values, encode_gamma(linear_values), label='gamma-encoded (stored pixel value)')
plt.xlabel('linear light intensity')
plt.ylabel('encoded value')
plt.legend(fontsize=8)
plt.title('Gamma encoding curve')
plt.show()

### Cạm bẫy: lấy trung bình các điểm ảnh đã mã hóa không phải là lấy trung bình ánh sáng

Trộn, đổi kích thước và làm mờ (Bài 8-10) đều ngầm giả định rằng một đại lượng tuyến tính đang được lấy trung bình. Giá trị điểm ảnh thường không tuyến tính &mdash; chúng đã được mã hóa gamma &mdash; nên lấy trung bình ngây thơ hai giá trị điểm ảnh đã mã hóa cho ra *độ sáng vật lý sai*, đôi khi sai nghiêm trọng.

In [ ]:
black_encoded, white_encoded = encode_gamma(0.0), encode_gamma(1.0)

# WRONG: average the encoded (stored) pixel values directly
naive_blend_encoded = (black_encoded + white_encoded) / 2
naive_blend_as_linear_light = decode_gamma(naive_blend_encoded)

# RIGHT: decode to linear light first, average there, then re-encode for storage/display
correct_blend_linear = (decode_gamma(black_encoded) + decode_gamma(white_encoded)) / 2
correct_blend_encoded = encode_gamma(correct_blend_linear)

print(f'true 50% linear-light gray, correctly encoded: {correct_blend_encoded:.3f}')
print(f'naive average of encoded black/white:           {naive_blend_encoded:.3f}')
print()
print(f'naive result represents only {naive_blend_as_linear_light:.1%} of the true linear brightness '
      f'(should be 50%)')

swatch = np.zeros((80, 240), dtype=np.float64)
swatch[:, :80] = naive_blend_encoded
swatch[:, 80:160] = correct_blend_encoded
swatch[:, 160:] = 0.5
plt.imshow(swatch, cmap='gray', vmin=0, vmax=1)
plt.title('naive blend  |  correct blend  |  encoded 0.5 for reference')
plt.axis('off')
plt.show()

Phép trộn ngây thơ tối hơn rõ rệt, đáng kể so với kết quả xám 50% đúng về mặt vật lý &mdash; một lỗi thực tế và phổ biến khi code xử lý ảnh (làm mờ, tạo mipmap, trộn alpha) thao tác trực tiếp trên giá trị điểm ảnh đã mã hóa gamma thay vì tuyến tính hóa trước.

## Cân bằng trắng: màu sắc phụ thuộc vào nguồn sáng

Camera ghi nhận *tích* của hệ số phản xạ của bề mặt và màu của nguồn chiếu sáng, chứ không chỉ màu "thực" của bề mặt &mdash; một tờ giấy trắng trông có màu cam dưới ánh đèn sợi đốt ấm và hơi xanh dưới bầu trời u ám, dù hệ thị giác của chúng ta thường (phần lớn) tự động bù trừ điều này (tính bất biến màu — color constancy). Camera phải thực hiện cùng phép hiệu chỉnh đó một cách có chủ đích: **cân bằng trắng (white balancing)**.

In [ ]:
rng = np.random.default_rng(0)
true_scene = np.zeros((150, 150, 3), dtype=np.uint8)
cv2.rectangle(true_scene, (20, 20), (130, 130), (150, 150, 150), -1)  # a neutral gray card
cv2.circle(true_scene, (75, 75), 30, (60, 120, 200), -1)
true_scene = np.clip(true_scene.astype(np.float64) + rng.normal(0, 5, true_scene.shape), 0, 255).astype(np.uint8)

tungsten_gains = np.array([1.3, 1.0, 0.6])  # BGR: boosts red, cuts blue -- a warm cast
cast_scene = np.clip(true_scene.astype(np.float64) * tungsten_gains, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(6, 3.5))
axes[0].imshow(cv2.cvtColor(true_scene, cv2.COLOR_BGR2RGB))
axes[0].set_title('True colors')
axes[1].imshow(cv2.cvtColor(cast_scene, cv2.COLOR_BGR2RGB))
axes[1].set_title('Under warm tungsten light')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Giả định thế giới xám

Một thuật toán cân bằng trắng đơn giản, kinh điển giả định rằng, khi lấy trung bình trên toàn bộ một cảnh thực tế, các màu gần như triệt tiêu nhau thành xám trung tính. Theo giả định đó, bất kỳ khác biệt có hệ thống nào giữa giá trị trung bình theo từng kênh của bức ảnh đều tiết lộ sắc thái màu của nguồn sáng &mdash; nên việc co giãn lại mỗi kênh để cân bằng các giá trị trung bình sẽ hoàn tác nó một cách gần đúng.

In [ ]:
def gray_world_white_balance(image):
    image_f = image.astype(np.float64)
    channel_means = image_f.reshape(-1, 3).mean(axis=0)
    target_gray = channel_means.mean()
    gains = target_gray / channel_means
    return np.clip(image_f * gains, 0, 255).astype(np.uint8)

corrected_scene = gray_world_white_balance(cast_scene)

error_before = np.abs(cast_scene.astype(int) - true_scene.astype(int)).mean()
error_after = np.abs(corrected_scene.astype(int) - true_scene.astype(int)).mean()
print(f'mean abs pixel error before correction: {error_before:.2f}')
print(f'mean abs pixel error after correction:  {error_after:.2f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [true_scene, cast_scene, corrected_scene],
                          ['True colors', 'Color cast', 'Gray-world corrected']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Phép hiệu chỉnh giảm sai số đáng kể, nhưng không về 0 &mdash; giả định thế giới xám ở đây chỉ gần đúng (cảnh nhỏ này không hoàn toàn trung tính khi lấy trung bình, vì nó bị chi phối bởi hình tròn có màu), đó chính là lý do camera thực thường kết hợp nó với các tín hiệu khác (một mảng tham chiếu xám/trắng thực sự được phát hiện, thống kê cảnh học được, hoặc các thiết lập sẵn do người dùng chỉ định) thay vì chỉ dựa vào giả định thế giới xám.

### Bài tập

1. Thay đổi `strength` trong `defocus_kernel_size` để làm hiệu ứng độ sâu trường ảnh nông hơn nhiều (`strength` lớn hơn) hoặc sâu hơn nhiều (`strength` nhỏ hơn). Với `strength=0`, điều gì nên xảy ra, và code có tạo ra điều đó không?
2. Sửa `make_bayer_rggb` và các phép gán `Rmask`/`Gmask`/`Bmask` trong `demosaick_bilinear_rggb` để dựng và giải mã một bức khảm BGGR thay vì RGGB (hoán đổi góc nào là đỏ và góc nào là lam). Tìm mã OpenCV tương ứng bằng cách thử lần lượt `cv2.COLOR_BayerBG2BGR`, `cv2.COLOR_BayerGB2BGR`, `cv2.COLOR_BayerGR2BGR`, `cv2.COLOR_BayerRG2BGR` và kiểm tra cái nào đưa MAE trở về mức nền nhiễu — quy ước đặt tên các mã này của OpenCV nổi tiếng là dễ bị hiểu ngược, bản thân điều đó đã là một minh họa nhỏ cho luận điểm của bài này về các lỗi không khớp mẫu âm thầm.
3. Lặp lại thí nghiệm trộn gamma, nhưng trộn `0.2` và `0.8` (thay vì đen và trắng thuần) theo cả cách ngây thơ lẫn cách đúng. Độ chênh lệch giữa chúng lớn hơn hay nhỏ hơn trường hợp đen/trắng, và tại sao điều đó hợp lý dựa trên hình dạng của đường cong gamma?
4. Sửa ví dụ cân bằng trắng để cảnh bị chi phối bởi một vùng *đỏ bão hòa* lớn thay vì một thẻ xám trung tính (tức là làm cho giả định cốt lõi của thế giới xám rõ ràng là sai). `gray_world_white_balance` có còn cải thiện kết quả, không tạo ra khác biệt, hay thậm chí làm nó tệ hơn?